In [ ]:
from pathlib import Path
import sys

project_root = Path(r'C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting')
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print('Project root:', project_root)
print('Features file exists:', (project_root / 'data' / 'processed' / 'train_features.parquet').exists())


In [ ]:
import pandas as pd
import numpy as np

features = pd.read_parquet(project_root / 'data' / 'processed' / 'train_features.parquet').sort_values('Date').copy()
features['Date'] = pd.to_datetime(features['Date'])
print('rows:', len(features))
print('date range:', features['Date'].min(), features['Date'].max())
print('stores:', features['Store'].nunique())
print('target summary:', features['Sales'].describe().to_string())


In [ ]:
from src.evaluation import rmspe

# Holdout dernier horizon: 6 semaines
split_date = features['Date'].max() - pd.Timedelta(weeks=6)
train_df = features[features['Date'] < split_date].copy()
test_df = features[features['Date'] >= split_date].copy()
print('train rows:', len(train_df))
print('test rows:', len(test_df))
print('test start:', test_df['Date'].min(), 'test end:', test_df['Date'].max())

baseline = train_df.groupby(['Store', 'day_of_week'])['Sales'].mean().reset_index().rename(columns={'Sales': 'sales_mean_by_store_day'})
combined = test_df.merge(baseline, on=['Store', 'day_of_week'], how='left')
combined['pred'] = combined['sales_mean_by_store_day'].fillna(train_df['Sales'].mean())

mae = np.mean(np.abs(combined['Sales'] - combined['pred']))
rmse = np.sqrt(np.mean((combined['Sales'] - combined['pred']) ** 2))
rmspe_value = rmspe(combined['Sales'], combined['pred'])

print('Baseline metrics:')
print(f'MAE={mae:.2f}')
print(f'RMSE={rmse:.2f}')
print(f'RMSPE={rmspe_value:.4f}')
